# Layer 2: two session types, no switching

Layer 1 gave every track the same skip rate. Its predictive checks showed that sessions differ a lot: some sessions have almost no skips, others almost only skips.

In this notebook each session has one hidden **type** for all its tracks: *Engaged* (few skips) or *Browsing* (many skips). The type never changes inside a session, so this model has differences between sessions but no switching. Layer 3 adds switching, and it must beat this model before we can say that listeners switch moods.

This is the "two customer segments" mixture of Chapter 4, with Bernoulli components instead of Normal ones.

In [ ]:
import json
import os
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import posterior as po
import session_types as st
from checks import (all_checks, average_log_score, load_sessions,
                    real_data_checks, split_users, to_matrix)

SEED = 2026
rng = np.random.default_rng(SEED)

N_DRAWS = 20_000     # importance sampling draws
DF = 4               # degrees of freedom of the Student-t proposal
N_POSTERIOR = 1000   # resampled posterior draws (predictive checks, test score)
N_STARTS = 10        # EM starting points

# Beta(2, 2) for theta_e, theta_b and p0 (section 2)
# EM uses Beta(3, 3) to find the mode on the log-odds scale (section 3)
PRIOR = st.PRIOR
EM_PRIOR = po.log_odds_prior(PRIOR)

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

BLUE = "#2a78d6"
ORANGE = "#eb6834"
GREY = "#52514e"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY,
    "xtick.color": GREY,
    "ytick.color": GREY,
    "axes.labelcolor": GREY,
})

The model functions are in `session_types.py` and the general tools (importance sampling, Hessian, predictive checks) in `posterior.py`. Layer 3 reuses `posterior.py` as it is.

## 1. The data

We use the same sessions and the same training/test split as Layer 1 (split by user, seed 2026). Layer 1's results are read from `results/layer1.json` for the comparisons.

In [ ]:
sessions = load_sessions()
train, test = split_users(sessions, seed=SEED)
train_skips, train_mask = to_matrix(train)
test_skips, test_mask = to_matrix(test)
train_lengths = train_mask.sum(axis=1)

with open("results/layer1.json") as file:
    layer1 = json.load(file)

print(f"Training: {len(train_skips)} sessions, {train_mask.sum()} tracks, "
      f"skip rate {train['skip'].mean():.4f}")
print(f"Test: {len(test_skips)} sessions, {test_mask.sum()} tracks")
print(f"Session length: {train_lengths.min()} to {train_lengths.max()} tracks, "
      f"mean {train_lengths.mean():.2f}")

The training data have about 4300 sessions and 67,000 tracks, with the same skip rate as in Layer 1. The test sessions are only used in section 8.

## 2. The model

**Symbols.** For session $i$ with $n_i$ tracks, $y_{it} = 1$ if track $t$ is skipped and $0$ if it is played.

- $z_i \in \{E, B\}$ is the hidden type of session $i$ (Engaged or Browsing). It is the same for all tracks of the session.
- $p_0 = P(z_i = E)$ is the share of Engaged sessions.
- $\theta_E$ and $\theta_B$ are the skip rates of the two types.

**Model.**

$$z_i \sim \text{Bernoulli}(p_0), \qquad y_{it} \mid z_i = E \sim \text{Bernoulli}(\theta_E), \qquad y_{it} \mid z_i = B \sim \text{Bernoulli}(\theta_B)$$

Given the type, tracks are independent. We do not know the type, so we add the two possibilities (law of total probability, Ch. 1):

$$p(y_i) = p_0\, \theta_E^{s_i} (1-\theta_E)^{n_i - s_i} + (1-p_0)\, \theta_B^{s_i} (1-\theta_B)^{n_i - s_i}$$

where $s_i$ is the number of skips. **Only $n_i$ and $s_i$ appear**: the product $\prod_t \theta^{y_{it}}(1-\theta)^{1-y_{it}}$ is the same for every order of the tracks. This is "no switching" written as a formula. (There is no binomial coefficient, because we observe the order; it simply does not change the probability.)

**Priors.**

| Parameter | Prior | Why |
|---|---|---|
| $\theta_E, \theta_B$ | Beta(2, 2), with $\theta_E < \theta_B$ | Weakly informative, keeps values away from 0 and 1. The order fixes label switching: "Engaged" always means the type with fewer skips (Ch. 4). |
| $p_0$ | Beta(2, 2) | No strong knowledge of how many sessions are engaged. |

Beta(2, 2) has the weight of 4 observations, while the data have about 4300 sessions and 67,000 tracks, so the prior matters very little (checked in section 9).

**Link with Layer 3.** Layer 3 lets the type change from track to track with probabilities $1-a$ and $1-b$. With $a = b = 1$ the type never changes, and Layer 3 becomes this model.

**First check, by hand.** With $\theta_E = 0.1$, $\theta_B = 0.7$, $p_0 = 0.5$, a session of two skips has likelihood $0.5 \times 0.1^2 + 0.5 \times 0.7^2 = 0.005 + 0.245 = 0.25$, and by Bayes' rule $P(B \mid \text{two skips}) = 0.245 / 0.25 = 0.98$.

In [ ]:
example = {"theta_e": 0.1, "theta_b": 0.7, "p0": 0.5}
two_skips = np.array([[1, 1]])
two_tracks = np.ones((1, 2), dtype=bool)

likelihood = np.exp(st.log_likelihood(example, two_skips, two_tracks))
p_browsing = 1 - st.type_probabilities(example, two_skips, two_tracks)[0]
print(f"Likelihood of two skips: {likelihood:.4f} (by hand 0.25)")
print(f"P(Browsing type | two skips): {p_browsing:.4f} (by hand 0.98)")

The code gives the two values found by hand. Two skips in a row are 49 times more likely for a Browsing session (0.49 against 0.01), so the posterior moves strongly towards Browsing.

**Second check, brute force.** We write every possible session of 1 to 4 tracks (30 sessions) in rows of 6 cells. The cells after the end of a session are filled with 1s, to check that the mask really hides them. For each session we multiply the probabilities of its tracks directly, without logs and without counts, and add the two types. The code works in logs with counts, so both routes must agree.

In [ ]:
# Every session of 1 to 4 tracks: the binary digits of 0, 1, ..., 2^length - 1
blocks = []
tiny_lengths = []
for length in range(1, 5):
    codes = np.arange(2 ** length)
    digits = (codes[:, None] >> np.arange(6)) & 1
    # junk 1s after the end of the session
    blocks.append(np.where(np.arange(6) < length, digits, 1))
    tiny_lengths.append(np.full(2 ** length, length))
tiny_skips = np.vstack(blocks)
tiny_lengths = np.concatenate(tiny_lengths)
tiny_mask = np.arange(6) < tiny_lengths[:, None]

q = {"theta_e": 0.23, "theta_b": 0.81, "p0": 0.37}
track_e = np.where(tiny_mask, np.where(tiny_skips == 1, q["theta_e"], 1 - q["theta_e"]), 1)
track_b = np.where(tiny_mask, np.where(tiny_skips == 1, q["theta_b"], 1 - q["theta_b"]), 1)
direct = q["p0"] * track_e.prod(axis=1) + (1 - q["p0"]) * track_b.prod(axis=1)

ours = np.array([np.exp(st.log_likelihood(q, tiny_skips[[i]], tiny_mask[[i]]))
                 for i in range(len(tiny_skips))])
print(f"{len(direct)} sessions, largest difference: {np.abs(ours - direct).max():.1e}")
print(f"Sum over the 16 sessions of 4 tracks: {direct[tiny_lengths == 4].sum():.6f}")

The two routes agree up to rounding error ($10^{-16}$), and the probabilities of all sessions of 4 tracks add up to 1, so the model is a proper probability distribution and the mask works.

## 3. Validation on simulated data

Before touching the real data, we fit the model on data where we know the truth. We simulate as many sessions as in the training data, **with the same lengths**.

True values: $\theta_E = 0.05$, $\theta_B = 0.85$, $p_0 = 0.55$. The overall skip rate is $0.55 \times 0.05 + 0.45 \times 0.85 = 0.41$, close to Yambda's 0.42. These values give many sessions with no skip and many with almost only skips, as in Yambda.

The simulator uses the discrete inverse transform (Ch. 2): a uniform number below $p_0$ makes the session Engaged, then a uniform below the session's skip rate makes a track skipped. We first check it on a large sample (50 times the training sessions).

In [ ]:
TRUE = {"theta_e": 0.05, "theta_b": 0.85, "p0": 0.55}
sim_skips, sim_mask, sim_engaged = st.simulate(TRUE, train_lengths, rng)

big_skips, big_mask, big_engaged = st.simulate(TRUE, np.repeat(train_lengths, 50),
                                               np.random.default_rng(SEED))
again = st.simulate(TRUE, np.repeat(train_lengths, 50), np.random.default_rng(SEED))
print("Same seed, same output:", np.array_equal(again[0], big_skips))
print("Same lengths as the training data:", np.array_equal(sim_mask, train_mask))

pd.DataFrame({
    "true": [0.55, 0.05, 0.85, 0.41],
    "simulated": [big_engaged.mean(),
                  big_skips[big_engaged][big_mask[big_engaged]].mean(),
                  big_skips[~big_engaged][big_mask[~big_engaged]].mean(),
                  big_skips[big_mask].mean()],
}, index=["share Engaged (p0)", "skip rate, Engaged", "skip rate, Browsing",
          "overall skip rate"]).round(4)

The simulator gives back the true values to the 2nd or 3rd decimal (the overall rate is a little below 0.41 only because this sample has slightly more Engaged sessions than 55%). The same seed gives the same sessions, and the simulated data have exactly the training lengths.

**Finding the mode with EM (Ch. 4).** EM alternates two steps:

- E-step: for each session, the probability $r_i$ that it is Engaged, by Bayes' rule.
- M-step: the Beta updates of Ch. 1, where session $i$ counts as Engaged with weight $r_i$. For example $\theta_E = (\sum_i r_i s_i + \alpha - 1)/(\sum_i r_i n_i + \alpha + \beta - 2)$: expected skips over expected tracks, plus the prior as pseudo-counts.

**Why EM runs with Beta(3, 3) and not Beta(2, 2).** Importance sampling (Ch. 3) works on the log-odds scale $\phi = \log(p/(1-p))$, where the parameters can take any real value. Changing the variable multiplies the density by the Jacobian $dp/d\phi = p(1-p)$ (Ch. 2). Beta(2, 2) times $p(1-p)$ is $p^2(1-p)^2$, which has the shape of Beta(3, 3). So EM with every prior raised by 1 finds **the mode on the log-odds scale**, which is where we centre the proposal. For one parameter with $s = 6000$ skips out of $n = 40{,}000$ tracks, the two modes are $6001/40002 = 0.150017$ and $6002/40004 = 0.150035$: a difference of 1% of a posterior standard deviation, but this way the centre is exact. We report only this EM result; the intervals come from importance sampling.

We start EM from 10 random points drawn from the prior, and check that the log-posterior never decreases and that every start ends at the same point.

In [ ]:
sim_starts = [st.random_start(rng) for _ in range(N_STARTS)]
sim_em, sim_mode, _ = po.em_from_starts(st.em, sim_starts, sim_skips, sim_mask, EM_PRIOR)
sim_em.round(6)

All 10 starts reach the same point in a few iterations, and the log-posterior never decreases. EM is fast here because almost every session clearly shows its type.

**Importance sampling (Ch. 3).** We want the posterior, not only its mode. We draw from a proposal $q$ that is easy to sample and weight each draw by $w = \text{posterior}/q$, in logs and normalized with log-sum-exp.

- Proposal: a Student-t with 4 degrees of freedom on the log-odds scale, centred at the mode, with scale matrix $(-H)^{-1}$, where $H$ is the Hessian (matrix of second derivatives) of the log-posterior at the mode. Near the mode the log-posterior is close to a parabola, and $(-H)^{-1}$ is the covariance of the matching Normal. The Student-t has the same centre and shape but heavier tails than the target, which keeps the weights bounded (Ch. 3, Prop. 3.3).
- Draws: a Normal divided by $\sqrt{\chi^2_4/4}$, with the Cholesky factor of the scale matrix (Ch. 2).
- The prior stays Beta; on the log-odds scale its density includes the Jacobian $\log p + \log(1-p)$ (Ch. 2, Ch. 4 Ex. 4.1). Draws with $\theta_E \ge \theta_B$ get weight 0.
- The Hessian is computed by central differences. The course does this in one dimension; here it is a 3 × 3 matrix.

In [ ]:
sim_log_post = partial(st.log_posterior, skips=sim_skips, mask=sim_mask, prior=PRIOR)
sim_fit = po.importance_sampling_at_mode(sim_log_post, po.to_vector(sim_mode, st.NAMES),
                                         N_DRAWS, rng, DF)
po.posterior_table(sim_fit, st.NAMES, truth=TRUE).round(5)

**All three true values are inside the 95% intervals.** The intervals are narrow because about 4300 sessions carry a lot of information. The Monte Carlo standard errors are about 100 times smaller than the posterior standard deviations, so 20,000 draws are more than enough.

**Diagnostics of the proposal.** The gradient should be about 0 at the centre (it is the mode), the Hessian should barely change when the finite-difference step is halved, and all eigenvalues of $-H$ should be positive (a real peak). Then we look at the weights: the effective sample size $\text{ESS} = 1/\sum_k \bar w_k^2$ and the largest weight (Ch. 3).

In [ ]:
po.proposal_diagnostics(sim_log_post, sim_fit).round(6)

The gradient is about 0, so the proposal is centred at the mode. The Hessian does not change when the step is halved, and all its eigenvalues are positive. No draw breaks θ_E < θ_B, and the largest weight is tiny.

The ESS is about **86.5%** of the draws. This is exactly what a Student-t(4) proposal gives when the target is a 3-dimensional Normal with the same centre and shape (checked by simulation). So the posterior is very close to Normal on the log-odds scale, and the 13.5% loss is the price of the heavy tails that keep the weights safe (Prop. 3.3).

**A hard simulated data set (identifiability).** The data set above is easy: with skip rates 0.05 and 0.85, a session of 10 tracks almost always shows its type. We now bring the two types closer: $\theta_E = 0.30$, $\theta_B = 0.55$, $p_0 = 0.55$.

**What we expect before running it:**

- More sessions are ambiguous, so EM needs more iterations (EM is slow when the hidden part holds much of the information, Ch. 4).
- Wider intervals, especially for $p_0$, since the types of many sessions are uncertain.
- Stronger correlations between parameters: the data can trade "more Engaged sessions" against "a higher Engaged skip rate".
- Possibly a lower ESS, if the posterior is less close to Normal on the log-odds scale.
- The true values should still be inside the 95% intervals.

In [ ]:
HARD = {"theta_e": 0.30, "theta_b": 0.55, "p0": 0.55}
hard_skips, hard_mask, hard_engaged = st.simulate(HARD, train_lengths, rng)

hard_starts = [st.random_start(rng) for _ in range(N_STARTS)]
hard_em, hard_mode, _ = po.em_from_starts(st.em, hard_starts, hard_skips, hard_mask, EM_PRIOR)
hard_log_post = partial(st.log_posterior, skips=hard_skips, mask=hard_mask, prior=PRIOR)
hard_fit = po.importance_sampling_at_mode(hard_log_post, po.to_vector(hard_mode, st.NAMES),
                                          N_DRAWS, rng, DF)
po.posterior_table(hard_fit, st.NAMES, truth=HARD).round(5)

The true values are again inside the 95% intervals, which are now clearly wider.

We compare the two simulated data sets side by side. "Sessions classified right" uses the true values: the share of sessions whose most likely type (probability above 0.5) is the true type.

In [ ]:
def correlation_of(fit):
    '''Return the posterior correlation between theta_e and theta_b'''
    return po.weighted_correlation(fit, st.NAMES).loc["theta_e", "theta_b"]

easy_table = po.posterior_table(sim_fit, st.NAMES)
hard_table = po.posterior_table(hard_fit, st.NAMES)
pd.DataFrame({
    "easy (0.05 / 0.85)": {
        "EM iterations (mean of 10 starts)": sim_em["iterations"].mean(),
        "width of the p0 interval": easy_table.loc["p0", "high"] - easy_table.loc["p0", "low"],
        "width of the theta_e interval": easy_table.loc["theta_e", "high"] - easy_table.loc["theta_e", "low"],
        "correlation theta_e, theta_b": correlation_of(sim_fit),
        "ESS share": sim_fit["ess_share"],
        "sessions classified right": ((st.type_probabilities(TRUE, sim_skips, sim_mask) > 0.5)
                                      == sim_engaged).mean(),
    },
    "hard (0.30 / 0.55)": {
        "EM iterations (mean of 10 starts)": hard_em["iterations"].mean(),
        "width of the p0 interval": hard_table.loc["p0", "high"] - hard_table.loc["p0", "low"],
        "width of the theta_e interval": hard_table.loc["theta_e", "high"] - hard_table.loc["theta_e", "low"],
        "correlation theta_e, theta_b": correlation_of(hard_fit),
        "ESS share": hard_fit["ess_share"],
        "sessions classified right": ((st.type_probabilities(HARD, hard_skips, hard_mask) > 0.5)
                                      == hard_engaged).mean(),
    },
}).round(4)

As expected, the hard data set needs many more EM iterations (about 110 instead of 5), its intervals are 2 to 4 times wider, θ_E and θ_B are now strongly correlated (0.6 instead of 0), and only 85% of sessions show their true type.

One expectation was wrong: **the ESS stays at 86%**. The ESS does not measure how wide or how correlated the posterior is, because the proposal's scale matrix $(-H)^{-1}$ adapts to both: a wider posterior gives a wider proposal, and correlations appear in the off-diagonal terms. The ESS only measures how far the posterior's *shape* is from the proposal's shape. With about 4300 sessions, even this harder posterior is still close to Normal on the log-odds scale (with many independent sessions, the posterior becomes Normal, as in a central limit theorem). So the ESS stays at the Student-t(4)-versus-Normal value of 86.5%. It would drop with much less data, or with a posterior pushed against a boundary, for example θ_E close to θ_B.

The full correlation matrix of the hard data set:

In [ ]:
po.weighted_correlation(hard_fit, st.NAMES).round(3)

All three parameters move together. If more sessions are called Engaged (p0 up), the extra ones are the borderline sessions with a middle skip rate: they raise θ_E, and the Browsing sessions that remain skip more, which raises θ_B.

## 4. Choice of the proposal

As in Ch. 3 Example 3.4, we compare three proposals on the easy simulated data set, with the same number of draws:

- the Student-t (4 df) at the mode, as above;
- the prior itself: Beta(2, 2) draws for each parameter, moved to the log-odds scale. About half of them break $\theta_E < \theta_B$;
- a Normal at the mode with **half** the standard deviation: its tails are lighter than the target's, which Prop. 3.3 warns against.

If no draw has a finite weight, the table says "no valid draw" instead of a number.

In [ ]:
center = sim_fit["center"]
narrow_cov = sim_fit["shape"] / 4   # half the standard deviation
prior_draws = po.prior_draws_log_odds(PRIOR, st.NAMES, N_DRAWS, rng)
narrow_draws = po.normal_draws(center, narrow_cov, N_DRAWS, rng)
proposals = {
    "Student-t (4 df) at the mode": (
        sim_fit["draws"], po.student_t_log_density(sim_fit["draws"], center, sim_fit["shape"], DF)),
    "prior Beta(2, 2)": (
        prior_draws, po.log_beta_prior_log_odds(prior_draws, PRIOR, st.NAMES)),
    "Normal at the mode, half the sd": (
        narrow_draws, po.normal_log_density(narrow_draws, center, narrow_cov)),
}

comparison = []
for name, (draws, log_proposal) in proposals.items():
    result = po.importance_sampling(sim_log_post, draws, log_proposal)
    theta_e = po.to_params(draws, st.NAMES)["theta_e"]
    comparison.append({
        "proposal": name,
        "draws with theta_e < theta_b": result["n_valid"],
        "ESS": round(result["ess"], 1),
        "ESS share": round(result["ess_share"], 5),
        "largest weight": round(result["largest_weight"], 4),
        "theta_e mean": (f"{(result['weights'] * theta_e).sum():.4f}"
                         if result["n_valid"] > 0 else "no valid draw"),
    })
comparison = pd.DataFrame(comparison)
comparison

This is the lesson of Example 3.4:

- The **Student-t at the mode** keeps about 87% of its draws useful.
- The **prior** spreads its draws over the whole space, while the posterior is a tiny region. One draw takes all the weight (ESS = 1), so its estimate is just one lucky draw. About half of its draws respect θ_E < θ_B, but that does not help.
- The **narrow Normal** is centred in the right place, so its estimate looks reasonable here, but its ESS is only a few percent and its largest weight is about 100 times that of the Student-t.

**Why the narrow Normal is unsafe (Ch. 3, Prop. 3.3).** Take one direction, with target sd $\sigma$ and proposal sd $\sigma_q$. The variance of the weights is finite only if $E_q[w^2] = \int p^2/q$ is finite, which for two Normals requires $\sigma_q^2 > \sigma^2/2$. Our narrow Normal has $\sigma_q^2 = \sigma^2/4 < \sigma^2/2$, so **its weights have infinite variance**. Most draws then get small weights, and a rare draw far in the tails gets a huge one. The ESS and the estimates depend on whether such a draw happens. We repeat the run with 5 seeds.

In [ ]:
narrow_runs = []
for seed in [1, 2, 3, 4, 5]:
    draws = po.normal_draws(center, narrow_cov, N_DRAWS, np.random.default_rng(seed))
    result = po.importance_sampling(sim_log_post, draws,
                                    po.normal_log_density(draws, center, narrow_cov))
    theta_e = po.to_params(draws, st.NAMES)["theta_e"]
    narrow_runs.append({"seed": seed, "ESS share": result["ess_share"],
                        "largest weight": result["largest_weight"],
                        "theta_e mean": (result["weights"] * theta_e).sum()})
pd.DataFrame(narrow_runs).round(5)

The ESS of the narrow Normal jumps from one seed to the next, by a factor of about 20 (from well under 1% to about 3%), and the largest weight goes up to about 0.17: one draw out of 20,000 carries 17% of the answer. The estimate of θ_E moves by a few ten-thousandths between seeds, about 50 times its Student-t Monte Carlo standard error. With infinite-variance weights, the usual standard error formula cannot be trusted, and a bad seed can give a very wrong answer. The Student-t, which has heavier tails than the target, avoids this.

**Stability.** We repeat the Student-t run with three other seeds. The estimates should agree within a few Monte Carlo standard errors.

In [ ]:
stability = []
for seed in [1, 2, 3]:
    fit = po.importance_sampling_at_mode(sim_log_post, center, N_DRAWS,
                                         np.random.default_rng(seed), DF)
    means = po.posterior_table(fit, st.NAMES)["mean"]
    stability.append({"seed": seed, "ESS share": fit["ess_share"],
                      "largest weight": fit["largest_weight"], **means.to_dict()})
pd.DataFrame(stability).round(5)

With three other seeds the Student-t estimates agree to the 4th decimal, and the ESS stays near 86.5%. The results do not depend on the seed.

## 5. Fit on the Yambda training data

We now fit the real training sessions, with the two contexts pooled. First EM from 10 random starts, with the same Beta(3, 3) as in section 3.

In [ ]:
starts = [st.random_start(rng) for _ in range(N_STARTS)]
em_table, mode, histories = po.em_from_starts(st.em, starts, train_skips, train_mask, EM_PRIOR)
em_table.round(6)

On Yambda all 10 starts reach the same mode, and the log-posterior never decreases. EM needs about 30 iterations, about 6 times more than on the easy simulated data: the real types are closer to each other, so more sessions are ambiguous.

The figure shows how far each EM run is from the final log-posterior, on a log scale. A straight line means the error shrinks by the same factor at every step, which is how EM usually converges (Ch. 4).

In [ ]:
best = em_table["log-posterior"].max()
fig, ax = plt.subplots(figsize=(6, 3.2))
for history in histories:
    gap = np.maximum(best - history, 1e-10)
    ax.semilogy(np.arange(len(history)), gap, color=BLUE, alpha=0.6, linewidth=1.5)
ax.set_xlabel("EM iteration")
ax.set_ylabel("Final log-posterior minus current")
ax.grid(axis="y", color="#e5e4e0", linewidth=0.8)
ax.set_axisbelow(True)
fig.savefig("figures/layer2_em_convergence.pdf")
plt.show()

Every run follows a straight line: the error is divided by about the same factor at each iteration, until it reaches $10^{-10}$.

**A start to avoid (Ch. 4 lab, Task 3).** If EM starts with $\theta_E = \theta_B$, both types explain every session equally well, so the E-step gives $r_i = p_0$ for every session, whatever its skips. Both types then see the same expected data. We start at $\theta_E = \theta_B = 0.4$, once with $p_0 = 0.5$ and once with $p_0 = 0.6$.

(The trap of the lab with Normal components, a standard deviation going to 0, cannot happen here: a Bernoulli probability is at most 1, so the likelihood of a session is at most 1, and the Beta pseudo-counts keep every $\theta$ strictly between 0 and 1.)

In [ ]:
symmetric = []
for p0 in [0.5, 0.6]:
    start = {"theta_e": 0.4, "theta_b": 0.4, "p0": p0}
    end, history = st.em(start, train_skips, train_mask, n_iter=5000, prior=EM_PRIOR)
    symmetric.append({"start p0": p0, **{k: float(v) for k, v in end.items()},
                      "iterations": len(history) - 1, "log-posterior": history[-1]})
print(f"Layer 1 posterior mean skip rate: {layer1['posterior']['mean']:.6f}")
pd.DataFrame(symmetric).round(6)

With $p_0 = 0.5$, EM stops after one step with θ_E = θ_B at **almost exactly Layer 1's skip rate** (the small difference comes from the pseudo-counts of the prior). This point is a trap, not the answer: with equal skip rates and $p_0 = 0.5$, nothing can make the two types different.

With $p_0 = 0.6$, θ_E and θ_B are still nearly equal after one step, but the prior breaks the tie: both types get the same 2 pseudo-skips out of 4 pseudo-tracks, while the Engaged type has more weight (0.6) than the Browsing type (0.4), so the prior pulls the lighter type a little more towards 0.5. EM grows this small difference and reaches the usual mode.

Random starts from the prior never have θ_E = θ_B exactly, so they avoid this trap.

**Importance sampling on Yambda**, with the same proposal and diagnostics as in section 3.

In [ ]:
log_post = partial(st.log_posterior, skips=train_skips, mask=train_mask, prior=PRIOR)
fit = po.importance_sampling_at_mode(log_post, po.to_vector(mode, st.NAMES), N_DRAWS, rng, DF)
diagnostics = po.proposal_diagnostics(log_post, fit)
diagnostics.round(6)

The same checks pass on the real data: the gradient is close to 0 (EM stops at a tolerance, so the centre is off by less than $10^{-6}$), the Hessian is stable and has positive eigenvalues, no draw breaks the constraint, and the ESS is again about 86%. As in simulation, the posterior is close to Normal on the log-odds scale.

## 6. Posterior summaries

Posterior means, standard deviations, 95% intervals and Monte Carlo standard errors (MCSE) of the self-normalized estimates (Robert & Casella, 2004). The share of Browsing sessions is $1 - p_0$; since it is computed for each draw, it comes with its own interval.

In [ ]:
summary = po.posterior_table(fit, st.NAMES)
browsing_share = 1 - po.to_params(fit["draws"], st.NAMES)["p0"]
summary.loc["share of Browsing sessions"] = pd.Series(po.weighted_summary(browsing_share, fit["weights"]))
summary.round(5)

**About 54% of sessions are Engaged, with a skip rate of about 0.11, and about 46% are Browsing, with a skip rate of about 0.76.** The intervals are narrow and the Monte Carlo errors are tiny.

**Correlations between parameters**, computed from the weighted draws. (The off-diagonal terms of the Hessian give *partial* correlations, with the third parameter held fixed; these are the plain posterior correlations.)

In [ ]:
correlation = po.weighted_correlation(fit, st.NAMES)
correlation.round(3)

θ_E and θ_B are positively correlated (about 0.4), and both are positively correlated with p0 (about 0.2). The reason is the one seen on the hard data set: if the posterior calls more sessions Engaged, the extra ones are borderline sessions with a middle skip rate, so both skip rates go up. These correlations are weaker than on the hard data set, because the real types are further apart.

The marginal posteriors, as weighted histograms of the importance sampling draws.

In [ ]:
draw_params = po.to_params(fit["draws"], st.NAMES)
labels = {"theta_e": r"$\theta_E$", "theta_b": r"$\theta_B$", "p0": r"$p_0$"}
fig, axes = plt.subplots(1, 3, figsize=(9, 2.8))
for ax, name in zip(axes, st.NAMES):
    ax.hist(draw_params[name], bins=60, weights=fit["weights"], density=True,
            histtype="stepfilled", color=BLUE, alpha=0.8)
    ax.set_xlabel(labels[name])
    ax.set_yticks([])
axes[0].set_ylabel("Posterior density")
fig.savefig("figures/layer2_posterior.pdf")
plt.show()

The three marginal posteriors are narrow and look like Normal curves, as the high ESS suggested.

**Posterior draws.** For the rest of the notebook we resample 1000 draws with probabilities equal to the weights (Ch. 3). They are then equally weighted draws from the posterior.

**Which sessions are Browsing?** For each session, $P(z_i = B \mid y_i)$ by Bayes' rule, averaged over the 1000 draws.

In [ ]:
posterior_draws = po.resample(fit["draws"], fit["weights"], N_POSTERIOR, rng)
draws_params = [po.to_params(phi, st.NAMES) for phi in posterior_draws]
print(f"{len(np.unique(posterior_draws, axis=0))} different draws out of {N_POSTERIOR}")

p_browsing = sum(1 - st.type_probabilities(p, train_skips, train_mask)
                 for p in draws_params) / N_POSTERIOR
print(f"Sessions with P(Browsing) above 0.9: {(p_browsing > 0.9).mean():.3f}")
print(f"Sessions with P(Browsing) below 0.1: {(p_browsing < 0.1).mean():.3f}")
print(f"Sessions in between: {((p_browsing >= 0.1) & (p_browsing <= 0.9)).mean():.3f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(p_browsing, bins=20, range=(0, 1), color=BLUE)
ax.set_xlabel("P(Browsing type | session)")
ax.set_ylabel("Training sessions")
fig.savefig("figures/layer2_type_probabilities.pdf")
plt.show()

Most sessions are clearly one type: about 95% have a probability below 0.1 or above 0.9, and only about 5% are in between. After 10 to 20 tracks, the number of skips almost always tells which group a session belongs to. Almost all resampled draws are different points, so resampling did not lose much variety.

## 7. Posterior predictive checks

For each of the 1000 posterior draws we simulate sessions with the training lengths and compute six statistics (Ch. 1, posterior predictive). The first three are those of Layer 1; the last three are new:

- **gap after shuffling**: the gap after a skip once the tracks are shuffled inside each session (20 shuffles). It keeps the differences between sessions but destroys the order;
- **within-session gap**: gap after a skip minus gap after shuffling, the part that comes from the order inside a session;
- **skip/play changes** per session.

**What we expect.** Given its type, a session's tracks are independent, so Layer 2 should give a within-session gap close to 0 (real data 0.10). At the EM mode, each of the 14.6 neighbouring pairs of a session changes with probability $2\theta(1-\theta)$, so the expected number of changes is about $14.6 \times [0.536 \times 2 \times 0.109 \times 0.891 + 0.464 \times 2 \times 0.765 \times 0.235] \approx 3.9$ (real data 2.74).

The real-data values use a fixed seed for the shuffles, so all layers report the same numbers.

In [ ]:
observed = real_data_checks(train_skips, train_mask)
simulated = po.predictive_checks(st.simulate, draws_params, train_lengths, rng, all_checks)
check_table = po.predictive_table(simulated, observed)
check_table.round(4)

As expected, the **within-session gap is about 0** (real data 0.10), and the model predicts **about 3.9 skip/play changes per session** (real 2.74), matching the value found by hand. Given its type, a session's tracks are independent, so Layer 2 cannot produce runs of skips. For every statistic, none of the 1000 simulations reaches the real value (or, for the changes, all are above it).

**One surprise:** the gap after shuffling is about 0.44, not the 0.51 of the real data. We expected Layer 2 to explain all the differences between sessions. It explains most of them, but not all: the spread of skip rates between sessions is also slightly too small. The figure further below shows why.

**Layers 1 and 2 side by side.** Layer 1's table comes from `results/layer1.json`, computed with the same statistics, the same number of simulations and the same real-data values.

In [ ]:
if "predictive_checks" not in layer1:
    raise KeyError("results/layer1.json has no predictive_checks: rerun layer1_baseline.ipynb first")
layer1_checks = pd.DataFrame(layer1["predictive_checks"]).T
assert np.allclose(layer1_checks["real data"], check_table["real data"])

pd.DataFrame({
    "real data": check_table["real data"],
    "Layer 1 mean": layer1_checks["simulated mean"],
    "Layer 2 mean": check_table["simulated mean"],
    "Layer 1 share ≥ real": layer1_checks["share at least as large"],
    "Layer 2 share ≥ real": check_table["share at least as large"],
}).round(4)

Layer 2 is much closer to the real data than Layer 1 for the four statistics that come from differences between sessions (spread, gap after a skip, longest run, changes). Neither layer produces any within-session gap. This leaves the 0.10 that Layer 3 must reproduce.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(9, 5))
for ax, name in zip(axes.ravel(), simulated.columns):
    ax.hist(simulated[name], bins=30, color=BLUE, label="simulated (Layer 2)")
    ax.axvline(observed[name], color=ORANGE, linewidth=2, label="real data")
    ax.set_xlabel(name.capitalize())
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.set_yticks([])
axes[0, 0].set_ylabel("Simulated data sets")
axes[1, 0].set_ylabel("Simulated data sets")
handles, legend_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, legend_labels, frameon=False, loc="upper center", ncol=2)
fig.subplots_adjust(hspace=0.5, wspace=0.25)
fig.savefig("figures/layer2_predictive_checks.pdf")
plt.show()

In every panel the real value (orange) is outside the simulated range (blue): Layer 2 is better than Layer 1, but every statistic still rejects it.

**Skip rate per session.** Layer 2 says that sessions come in two groups. We compare the distribution of the skip rate per session in the real training data with the one simulated from 20 posterior draws.

In [ ]:
def session_rates(skips, mask):
    '''Return the skip rate of each session'''
    return (skips * mask).sum(axis=1) / mask.sum(axis=1)

real_rates = session_rates(train_skips, train_mask)
simulated_rates = np.concatenate([session_rates(*st.simulate(p, train_lengths, rng)[:2])
                                  for p in draws_params[:20]])

# bins centred on 0, 0.1, ..., 1, so sessions with 0% or 100% skips have their own bin
edges = np.linspace(-0.05, 1.05, 12)
real_share = np.histogram(real_rates, bins=edges)[0] / len(real_rates)
simulated_share = np.histogram(simulated_rates, bins=edges)[0] / len(simulated_rates)
centres = np.linspace(0, 1, 11)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(centres - 0.02, real_share, width=0.04, color=ORANGE, label="real data")
ax.bar(centres + 0.02, simulated_share, width=0.04, color=BLUE, label="simulated (Layer 2)")
ax.set_xlabel("Skip rate of the session")
ax.set_ylabel("Share of sessions")
ax.legend(frameon=False)
fig.savefig("figures/layer2_skip_rate_per_session.pdf")
plt.show()

rate_groups = {
    "no skip": lambda r: r == 0,
    "at most 10% skips": lambda r: r <= 0.1,
    "30% to 70% skips": lambda r: (r >= 0.3) & (r <= 0.7),
    "at least 90% skips": lambda r: r >= 0.9,
    "only skips": lambda r: r == 1,
}
rate_table = pd.DataFrame({
    "real data": {name: rule(real_rates).mean() for name, rule in rate_groups.items()},
    "simulated (Layer 2)": {name: rule(simulated_rates).mean() for name, rule in rate_groups.items()},
})
rate_table.round(3)

The real sessions are more varied than two types can make them. The real data have **more sessions with no skip** (22% against 10%), **more sessions with only skips** and **more sessions in the middle** (30% to 70% skips). Layer 2 instead puts too many sessions near its two skip rates. With θ_E ≈ 0.11 and about 15 tracks, an Engaged session has no skip with probability $0.89^{15.6} \approx 0.17$, so only about $0.54 \times 0.17 \approx 9\%$ of sessions have no skip, as simulated.

This is why the gap after shuffling is too small: real sessions differ from one another more than two fixed skip rates allow. Part of the middle sessions may come from switching inside a session (Layer 3), which mixes engaged and browsing tracks. The extremes (0% and 100%) point to differences between listeners that two types cannot capture.

## 8. Score on the test users

As in Layer 1, the score is the mean log probability given to what really happened on each test track. Unlike Layer 1, Layer 2 learns from the earlier tracks of the session: before track $t$, Bayes' rule updates the type probability with the $n_{<t}$ tracks and $s_{<t}$ skips seen so far (cumulative sums, no loop over tracks):

$$P(\text{skip at } t \mid y_{<t}) = P(E \mid y_{<t})\,\theta_E + P(B \mid y_{<t})\,\theta_B$$

At the first track this is $p_0 \theta_E + (1-p_0)\theta_B$. We average this probability over the 1000 posterior draws (the posterior predictive, Ch. 1), and also compute it at the posterior mean as a check.

In [ ]:
predictive = sum(st.skip_probabilities(p, test_skips, test_mask)
                 for p in draws_params) / N_POSTERIOR
score = average_log_score(predictive, test_skips, test_mask)

posterior_mean = {name: summary.loc[name, "mean"] for name in st.NAMES}
score_at_mean = average_log_score(st.skip_probabilities(posterior_mean, test_skips, test_mask),
                                  test_skips, test_mask)

print(f"Layer 2, averaged over the posterior: {score:.4f}")
print(f"Layer 2, at the posterior mean:       {score_at_mean:.4f}")
print(f"Layer 1, one skip rate:               {layer1['score_one_rate']:.4f}")
print(f"Fair coin:                            {np.log(0.5):.4f}")

**Layer 2 is much better than Layer 1 on new users: −0.479 against −0.678** (fair coin −0.693). After a few tracks the model knows the type of the session and predicts the next tracks much better. Averaging over the posterior or using the posterior mean gives the same score, because the posterior is very narrow.

## 9. Sensitivity to the prior

We repeat the fit (EM, then importance sampling) with Beta(1, 1) and Beta(5, 5) for all three parameters. For each prior, EM uses that prior raised by 1 (`log_odds_prior`), so the proposal is always centred at the right log-odds mode.

In [ ]:
priors = {"Beta(1, 1)": (1, 1), "Beta(2, 2), main": (2, 2), "Beta(5, 5)": (5, 5)}
sensitivity = []
for label, (alpha, beta) in priors.items():
    prior = {name: (alpha, beta) for name in st.NAMES}
    prior_mode, _ = st.em(st.random_start(rng), train_skips, train_mask,
                          prior=po.log_odds_prior(prior))
    prior_log_post = partial(st.log_posterior, skips=train_skips, mask=train_mask, prior=prior)
    prior_fit = po.importance_sampling_at_mode(prior_log_post, po.to_vector(prior_mode, st.NAMES),
                                               N_DRAWS, rng, DF)
    table = po.posterior_table(prior_fit, st.NAMES)[["mean", "low", "high"]]
    table.insert(0, "prior", label)
    table["ESS share"] = prior_fit["ess_share"]
    sensitivity.append(table)
sensitivity = pd.concat(sensitivity)
sensitivity.round(5)

The three priors give the same posterior means and intervals to the 3rd decimal, and the ESS stays at about 86%. With about 4300 sessions, the prior has almost no effect.

## 10. Save the results

Layer 3 reads this file to compare the three layers. The predictive-check table has the same six rows and the same columns as in `results/layer1.json`.

In [ ]:
def table_to_dict(table):
    '''Turn a DataFrame into a dict that json can write'''
    return json.loads(table.to_json(orient="index"))

results = {
    "em_mode_log_odds": {name: float(value) for name, value in mode.items()},
    "em_iterations": em_table["iterations"].tolist(),
    "posterior": table_to_dict(summary),
    "correlation": table_to_dict(correlation),
    "importance_sampling": {"draws": N_DRAWS, "df": DF,
                            **table_to_dict(diagnostics.to_frame("value"))},
    "proposal_comparison": json.loads(comparison.to_json(orient="records")),
    "simulated_easy": {"true": TRUE,
                       "posterior": table_to_dict(po.posterior_table(sim_fit, st.NAMES, TRUE))},
    "simulated_hard": {"true": HARD,
                       "posterior": table_to_dict(po.posterior_table(hard_fit, st.NAMES, HARD))},
    "share_sessions_browsing_above_0.9": float((p_browsing > 0.9).mean()),
    "observed_checks": {name: float(value) for name, value in observed.items()},
    "predictive_checks": table_to_dict(check_table),
    "session_rate_groups": table_to_dict(rate_table),
    "score_posterior_predictive": float(score),
    "score_posterior_mean": float(score_at_mean),
    "prior_sensitivity": json.loads(sensitivity.reset_index(names="parameter")
                                    .to_json(orient="records")),
}
with open("results/layer2.json", "w") as file:
    json.dump(results, file, indent=2)
print(list(results))

Every number used in the report for Layer 2 is now in `results/layer2.json`, and the figures are in `figures/layer2_*.pdf`.